# 🚀 Fine-Tuning Qwen 2.5 (7B) with Unsloth & QLoRA
This notebook demonstrates how to fine-tune **Qwen2.5-7B-Instruct** on Persian Question-Answering data using **Unsloth** and **QLoRA**, followed by exporting the model into **GGUF (Q4_K_M)** format for fast, local inference (via Ollama or llama.cpp).

### 📦 Step 1: Install Required Packages

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth
!pip install --no-deps "xformers" "trl" peft accelerate bitsandbytes


### 🧠 Step 2: Load 4-bit Quantized Qwen 2.5 Base Model

In [ ]:
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
max_seq_length = 1024

print(f"Loading {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)


### ⚙️ Step 3: Configure and Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA adapters successfully configured.")


### 📄 Step 4: Load and Format Persian Dataset into ChatML Template

In [ ]:
import json
from datasets import Dataset

# Load train and validation QA datasets
with open("qa_train.jsonl", "r", encoding="utf-8") as f:
    train_json = [json.loads(line) for line in f if line.strip()]

with open("qa_val.jsonl", "r", encoding="utf-8") as f:
    val_json = [json.loads(line) for line in f if line.strip()]

def format_data(examples):
    texts = []
    for sample in examples:
        messages = sample.get("messages", [])
        if not messages:
            messages = [
                {"role": "system", "content": sample.get("instruction", "تو یک دستیار پژوهشی هستی.")},
                {"role": "user", "content": sample.get("input", "")},
                {"role": "assistant", "content": sample.get("output", "")}
            ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_json).map(format_data, batched=True)
val_dataset = Dataset.from_list(val_json).map(format_data, batched=True)
print(f"Prepared {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")


### 🚀 Step 5: Supervised Fine-Tuning with SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
        eval_strategy="steps",
        eval_steps=10,
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")


### 💾 Step 6: Export to GGUF for Local Inference

In [ ]:
# Convert and save the trained model to GGUF format (Q4_K_M quantization)
model.save_pretrained_gguf("finetuned_model", tokenizer, quantization_method="q4_k_m")

from google.colab import files
files.download("finetuned_model/unsloth.Q4_k_m.gguf")
print("Model saved and download triggered!")
